### Use NLTK to implement text preprocessing techniques

In [2]:
# import packages
import pandas as pd
import tensorflow as tf
from textblob import TextBlob
from wordsegment import load, segment
from autocorrect import Speller
load()
import nltk
from nltk.corpus import stopwords
import emoji
import contractions
import re
import string
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('universal_tagset')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\stard\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_dat

True

### Load dataset

In [7]:
import pandas as pd
ds_name = 'kaggle_6class_3000'# 'offenseval'
path = f'preprocessed_datasets/{ds_name}'
train_DON = pd.read_csv(f'{path}/train/{ds_name}_train_DON.csv')
test_DON = pd.read_csv(f'{path}/test/{ds_name}_test_DON.csv')
val_DON = pd.read_csv(f'{path}/val/{ds_name}_val_DON.csv')
print(train_DON.shape)
print(test_DON.shape)
print(val_DON.shape)

(2098, 9)
(600, 9)
(298, 9)


### preprocessing functions definitions

In [ ]:
# Implemented by the paper, i.e. survey of preprocessing techniques.
# DON, LOW, RSW
# Do-Nothing preprocessing function.
def DON(input_data):
  tag_open_CDATA_removed = tf.strings.regex_replace(input_data, '<\!\[CDATA\[', ' ')
  tag_closed_CDATA_removed = tf.strings.regex_replace(tag_open_CDATA_removed,'\]{1,}>', ' ')
  tag_author_lang_en_removed = tf.strings.regex_replace(tag_closed_CDATA_removed,'<author lang="en">', ' ')
  tag_closed_author_removed = tf.strings.regex_replace(tag_author_lang_en_removed,'</author>', ' ')
  tag_open_documents_removed = tf.strings.regex_replace(tag_closed_author_removed,'<documents>\n(\t){0,2}', '')
  output_data = tf.strings.regex_replace(tag_open_documents_removed,'</documents>\n(\t){0,2}', ' ')
  return output_data

# Lowercasing preprocessing function.
def LOW(input_data):
  return tf.strings.lower(input_data).numpy().decode("utf-8")

# Removing Stop Words function.
def RSW(input_data):
  output_data = input_data
  try:
    input_string=output_data[0]
  except:
    input_string=output_data
    try:
      input_string = input_string.numpy()
    except:
      return output_data
    else:
      input_string=(str(input_string))[2:-1]
    blob = TextBlob(str(input_string)).words
    # print(blob)
    outputlist = [word for word in blob if word not in stopwords.words('english')]
    output_string = (' '.join(word for word in outputlist))
    output_tensor=tf.constant(output_string)
    return output_tensor.numpy().decode("utf-8")
  else:
    try:
      input_string = input_string.numpy()
    except:
      return output_data
    else:
      input_string=(str(input_string))[2:-1]
    blob = TextBlob(str(input_string)).words
    outputlist = [word for word in blob if word not in stopwords.words('english')]
    output_string = (' '.join(word for word in outputlist))
    output_tensor=tf.constant([[output_string]])
    return output_tensor.numpy().decode("utf-8")
  return output_data.numpy().decode("utf-8")

# Porter Stemmer preprocessing function.
def STM(input_data):
  output_data = input_data
  stemmer = nltk.PorterStemmer()
  try:
    input_string=output_data[0]
  except:
    input_string=output_data
    try:
      input_string = input_string.numpy()
    except:
      return output_data
    else:
      input_string=(str(input_string))[2:-1]
    blob = TextBlob(str(input_string)).words
    outputlist = [stemmer.stem(word) for word in blob]
    output_string = (' '.join(word for word in outputlist))
    output_tensor=tf.constant(output_string)
    return output_tensor.numpy().decode("utf-8")
  else:
    try:
      input_string = input_string.numpy()
    except:
      return output_data
    else:
      input_string=(str(input_string))[2:-1]
    blob = TextBlob(str(input_string)).words
    outputlist = [stemmer.stem(word) for word in blob]
    output_string = (' '.join(word for word in outputlist))
    output_tensor=tf.constant([[output_string]])
    return output_tensor.numpy().decode("utf-8")
  return output_data.numpy().decode("utf-8")


def STM_NLTK(text):
    tokens = nltk.word_tokenize(text)
    stemmer = nltk.PorterStemmer()
    outputlist = [stemmer.stem(t) for t in tokens]
    return ' '.join(outputlist)
# using NLTK functions to preform the preprocessing (10 techniques).

In [2]:
# all functions should take a string as input and return a string as output

def reg_preprocess(word):
    # Remove hashtags from words
    word = re.sub(r'#([^ ]*)', '#HASHTAG', word)
    # Remove consecutive periods in text
    word = re.sub(r'\.{2,}', '.', word)
    # Replace URLs (with or without HTTP/HTTPS both in upper and lower casaes) and 'www.' with keyword 'URL'
    word = re.sub('https?://[^\s/$.?#].[^\s]*|www\.[^\s/$.?#].[^\s]*','URL',word)
    word = re.sub('HTTPS?://[^\s/$.?#].[^\s]*|WWW\.[^\s/$.?#].[^\s]*','URL',word)
    # Replace mentions (@username) with '@USER'
    # word = re.sub(r'\B@(\w+)', '@USER', word)
    # Convert emojis to text
    word = emoji.demojize(word)
    # Add spaces around emojis
    word = re.sub(r'(:[^:]*:)', r' \1 ', word)
    # Add space before and after colon in 'word:word' format
    word = re.sub(r'(\w+):(\w+)', r'\1 : \2', word)
    # Remove extra spaces
    word = re.sub('\s{2,}', ' ', word)
    # Remove extra blank lines
    word = re.sub('\n{2,}','\n', word)
    # Replace extra exclamation marks with a single one
    word = re.sub(r'!{2,}','!',word)
    return word
# split
# remove URL / convert emoji to text / remove blank lines and extra spaces
def RNS(word):
    # Remove hashtags from words
    word = re.sub(r'#([^ ]*)', '#HASHTAG', word)
    # # Remove consecutive periods in text
    # word = re.sub(r'\.{2,}', '.', word)
    # Replace URLs (with or without HTTP/HTTPS both in upper and lower casaes) and 'www.' with keyword 'URL'
    word = re.sub('https?://[^\s/$.?#].[^\s]*|www\.[^\s/$.?#].[^\s]*','URL',word)
    word = re.sub('HTTPS?://[^\s/$.?#].[^\s]*|WWW\.[^\s/$.?#].[^\s]*','URL',word)
    # Replace mentions (@username) with '@USER'
    word = re.sub(r'\B@(\w+)', '@USER', word)
    # Convert emojis to text
    word = emoji.demojize(word)
    # Add spaces around emojis
    word = re.sub(r'(:[^:]*:)', r' \1 ', word)
    # Add space before and after colon in 'word:word' format
    word = re.sub(r'(\w+):(\w+)', r'\1 : \2', word)
    # Remove extra spaces
    word = re.sub('\s{2,}', ' ', word)
    # Remove extra blank lines
    word = re.sub('\n{2,}','\n', word)
    return word

def remove_spaces(text):
    '''
    Remove extra blank lines and spaces
    Preliminary of preprocessing
    Must be applied on text before other preprocessing techniques.
    '''
    # Remove extra spaces
    text = re.sub('\s{2,}', ' ', text)
    # Remove extra blank lines
    text = re.sub('\n{2,}','\n', text)
    return text


def EMOJI(text):
    text = emoji.demojize(text)
    # Add spaces around emojis
    text = re.sub(r'(:[^:]*:)', r' \1 ', text)
    return text

def RMURL(text):
    text = re.sub('https?://[^\s/$.?#].[^\s]*|www\.[^\s/$.?#].[^\s]*','URL',text)
    text = re.sub('HTTPS?://[^\s/$.?#].[^\s]*|WWW\.[^\s/$.?#].[^\s]*','URL',text)
    return text

def RCT(text: str) -> str:
    return contractions.fix(text)

def RRP(text: str) -> str:
    tokens = nltk.WordPunctTokenizer().tokenize(text)
    cleaned_tokens = []
    for token in tokens:
        # Replace repeated punctuation (e.g., "!!" -> "!", "..." -> ".")
        token = re.sub(r'([^\w\s])\1+', r'\1', token)
        cleaned_tokens.append(token)
    return nltk.TreebankWordDetokenizer().detokenize(cleaned_tokens)


def RPT_1(text: str) -> str:
    # Create translator object to remove all punctuation
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def RPT(text):
    tokenizer = nltk.RegexpTokenizer(r'\w+')
    detokenizer = nltk.TreebankWordDetokenizer()
    outputlist = tokenizer.tokenize(text)
    return detokenizer.detokenize(outputlist)

def RNB_1(text):
    # Create translator object to remove all numbers
    translator = str.maketrans('', '', '0123456789')
    return text.translate(translator)

def RNB(text: str) -> str:
    tokenizer = nltk.RegexpTokenizer(r'\D+')
    outputlist = tokenizer.tokenize(text)
    detokenizer = nltk.TreebankWordDetokenizer()
    return detokenizer.detokenize(outputlist)

def POS(text: str) -> str:
    # Tokenize and POS tag the text
    tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    words = []
    # ignore the punctuations
    for word, tag in tagged_tokens:
        words.append(word)
        if tag == '.':
            continue
        words.append(f'({tag})')
    detokenizer = nltk.TreebankWordDetokenizer()
    return  detokenizer.detokenize(words)

def LEM(text: str) -> str:
    lemmatizer = nltk.WordNetLemmatizer()
    tokens = nltk.word_tokenize(text)
    outputlist = [lemmatizer.lemmatize(t) for t in tokens]
    detokenizer = nltk.TreebankWordDetokenizer()
    return detokenizer.detokenize(outputlist)

def WSG(text):
    outputlist = segment(text)
    detokenizer = nltk.TreebankWordDetokenizer()
    return detokenizer.detokenize(outputlist)

def SCO(text):
    spell = Speller('en')
    return spell(text)

import scrubadub
import scrubadub_spacy

# Specify the spaCy model name
model_name = 'en_core_web_lg'

# Create the scrubber and add the spaCy detector
scrubber = scrubadub.Scrubber()
#scrubber.add_detector(scrubadub_spacy.detectors.SpacyEntityDetector(model="en_core_web_lg"))
scrubber.add_detector(scrubadub_spacy.detectors.SpacyEntityDetector(model="en_core_web_lg"))
# remove specific detectors
scrubber.remove_detector(scrubadub.detectors.EmailDetector)

def ANO(x):
    return scrubber.clean(x)

def ECR(text: str) -> str:
    pass

def RSA(text: str) -> str:
    pass

def EMO(text: str) -> str:
    pass

def NEG(text: str) -> str:
    pass

# not to write techniques by myself but use NLTK, find some functions for implementation
# reliable software

### Prepare and Save datasets using different preprocessing functions

In [8]:
import os
preprocess_functions = [ANO]
key = 'comment_text'
os.makedirs(f'{path}/train', exist_ok=True)
os.makedirs(f'{path}/test', exist_ok=True)
os.makedirs(f'{path}/val', exist_ok=True)
for func in preprocess_functions:
    print(f'Applying {func.__name__}...')
    tmp_train = train_DON.copy()
    tmp_train[key] = tmp_train[key].apply(func)
    tmp_train.to_csv(f'{path}/train/{ds_name}_train_{func.__name__}.csv', index=False)

    tmp_test = test_DON.copy()
    tmp_test[key] = tmp_test[key].apply(func)
    tmp_test.to_csv(f'{path}/test/{ds_name}_test_{func.__name__}.csv', index=False)

    tmp_val = val_DON.copy()
    tmp_val[key] = tmp_val[key].apply(func)
    tmp_val.to_csv(f'{path}/val/{ds_name}_val_{func.__name__}.csv', index=False)

print('Done!')

Applying ANO...
Done!


### testing

In [ ]:
test_data = ["I'd tap that and go back for seconds",
             'CLOSE YOUR FUCKING MOUTH!!!!!',
             "Fee fees and scooty puffs/poofs have become part of my normal vernacular. My husband thinks I'm clever and came up with those on my own...whoops.",
             'Well her hips do have like 3 weird curves of fat hanging off of them.',
             'Happens when skin folds onto itself. HAES, amirite?',
             "In order for her to get morbidly fat she'd have to lose weight, which sure as shit isn't gonna happen",
             "awesome- that's my week made :)",
             "I have two of these shirts and nothing to match them. Fuck it, I'm just gonna wear one as a skirt.",
             "I notice this as a waitress. Whenever someone (not just overweight but SHAMU sized) comes into the restaurant to eat, complains they can't sit where they want because they don't fit into the booth and complains about everything from the service to the lighting to the music, etc...How, just HOW can they complain about the food? I highly doubt that you've got high standards for food when you're wearing Wal-mart clothes and are, conservatively, at least 400+ lbs. And yet they complain. Maybe it's because our food doesn't taste like McDonalds but they'll order 3 or 4 things for themselves, complain about the food, eat all of the aforementioned food, and leave a &lt;10% tip. It's not just occasionally but usually.",
             'On any gay dating site: if they whine about how shallow guys are (and possess the MySpace angle) they are very likely to be an obeast. Pretty much the same as anywhere else.']

for i in range(10):
    sample = test_data[i]
    print('Ori:', sample)
    print('DON:', DON(sample))
    print('LOW:', LOW(sample))
    print('STM:', STM(sample))
    print('RSW:', RSW(sample))
    print('RNS:', RNS(sample))
    print('RCT:', RCT(sample))
    print('RRP:', RRP(sample))
    print('RPT:', RPT(sample))
    print('RNB:', RNB(sample))
    print('LEM:', LEM(sample))
    print('POS:', POS(sample))
    # print('ECR:', ECR(sample))
    print('WSG:', WSG(sample))
    print('SCO:', SCO(sample))
    print()


Ori: I'd tap that and go back for seconds
DON: I'd tap that and go back for seconds
LOW: i'd tap that and go back for seconds
STM: i 'd tap that and go back for second
RSW: I 'd tap go back seconds
RNS: I'd tap that and go back for seconds
RCT: I would tap that and go back for seconds
RRP: I' d tap that and go back for seconds
RPT: I d tap that and go back for seconds
RNB: I'd tap that and go back for seconds
LEM: I'd tap that and go back for second
POS: I (PRON)'d (VERB) tap (VERB) that (DET) and (CONJ) go (VERB) back (ADV) for (ADP) seconds (NOUN)
ECR: None
WSG: id tap that and go back for seconds
SCO: I'd tap that and go back for seconds

Ori: CLOSE YOUR FUCKING MOUTH!!!!!
DON: CLOSE YOUR FUCKING MOUTH!!!!!
LOW: close your fucking mouth!!!!!
STM: close your fuck mouth
RSW: CLOSE YOUR FUCKING MOUTH
RNS: CLOSE YOUR FUCKING MOUTH!!!!!
RCT: CLOSE YOUR FUCKING MOUTH!!!!!
RRP: CLOSE YOUR FUCKING MOUTH!
RPT: CLOSE YOUR FUCKING MOUTH
RNB: CLOSE YOUR FUCKING MOUTH!!!!!
LEM: CLOSE YOUR FUCKIN